# Fixedpoint ― 用递归规则求解可达性

## 本章要点

Z3 的 `Fixedpoint` 是处理**递归规则**的引擎。“从 A 能到 B，从 B 能到 C，那么从 A 能到 C”这样的推理，可以原样写成规则。

- 用规则（`rule`）和事实（`fact`）描述关系
- 用 `query()` 询问“是否成立”
- 能到达时，怎样取出实际的路径

默认引擎是**自底向上的 Datalog**。Datalog 类似关系数据库的查询（SQL 的递归查询）：从给定的事实出发反复应用规则，不断增加新的事实。

In [1]:
from z3 import *

## 例 1：图的可达性

在下面这个无向图中，调查从顶点 1 能否到达顶点 6。

```{mermaid}
flowchart LR
1 --- 2
1 --- 5
5 --- 2
4 --- 5
2 --- 3
4 --- 3
4 --- 6
```

用到 3 个谓词。

- `edge(a, b)` … `a` 和 `b` 之间有边（作为事实给出）
- `connect(a, b)` … 从 `a` 能直接移动到 `b`（有边则双向成立）
- `path(a, b)` … 从 `a` 能到达 `b`（递归地决定）

In [2]:
fp = Fixedpoint()
fp.set(engine='datalog')          # 使用 Datalog 引擎

s = BitVecSort(8)                  # 顶点编号用 8 位表示

connect = Function('connect', s, s, BoolSort())
edge = Function('edge', s, s, BoolSort())
path = Function('path', s, s, BoolSort())

a, b, c = Consts('a b c', s)

fp.register_relation(path, edge, connect)
fp.declare_var(a, b, c)

# 规则：有边就能直接移动（无向，所以两个方向都成立）
fp.rule(connect(a, b), edge(a, b))
fp.rule(connect(a, b), edge(b, a))

# 规则：能直接移动，就能到达
fp.rule(path(a, b), connect(a, b))
# 规则：a→b 能移动，且 b→c 能到达，则 a→c 能到达
fp.rule(path(a, c), [connect(a, b), path(b, c)])

# 事实：图的边
edges = [(1, 2), (1, 5), (5, 2), (4, 5), (2, 3), (4, 3), (4, 6)]
for p, q in edges:
    fp.fact(edge(p, q))

### 代码说明

**1. 引擎**

```python
fp = Fixedpoint()
fp.set(engine='datalog')
```

用 `engine='datalog'` 选择 Datalog 引擎。

**2. 顶点的类型**

```python
s = BitVecSort(8)
```

顶点编号用 8 位的位向量表示。Datalog 需要有限的类型，所以用 `BitVecSort` 这样的有限类型，而不是 `IntSort`。

**3. 定义谓词（关系）**

```python
connect = Function('connect', s, s, BoolSort())
```

用 `Function(名称, 参数的类型..., 返回值的类型)` 创建谓词。它是“接收两个顶点、返回真假的关系”，所以返回值是 `BoolSort()`。

**4. 声明变量、注册谓词**

```python
fp.register_relation(path, edge, connect)
fp.declare_var(a, b, c)
```

用 `register_relation()` 把谓词注册到引擎，用 `declare_var()` 声明规则里用的变量。

**5. 规则与事实**

```python
fp.rule(path(a, c), [connect(a, b), path(b, c)])
```

`rule(头, 本体)` 是“本体全部成立，则头也成立”的规则。这条规则读作“从 `a` 能移到 `b`，且从 `b` 能到达 `c`，那么从 `a` 能到达 `c`”。把自身（`path`）放进本体，就是递归。

`fp.fact(edge(1, 2))` 把“有边”注册为事实。

### 查询

In [3]:
print(fp.query(path(1, 6)))     # 能否到达
print(fp.query(path(1, 7)))     # 不存在的顶点

sat
unsat


从顶点 1 能到达 6（`sat`），到顶点 7 不能到达（`unsat`）。回答这种“是否成立”，就是 Fixedpoint 的工作。

## 重建路径

“能到达”已经知道了，但**该怎么走**还不知道。

Z3 的 Datalog 有输出**说明（explanations）**的功能，说明某个事实为什么成立（`datalog.generate_explanations`）。但这个功能在最近的 Z3（本环境是 5.1.0）里不稳定：会报 `explanations are not supported with undefined predicates`，或者答案变成 `unsat`，甚至让 Z3 整个崩溃。

所以本章**自己构造路径**。是否可达去问 Fixedpoint，具体的路径把边列表做 BFS（广度优先搜索）就能求出。

In [4]:
from collections import deque


def bfs_route(edges, start, goal, undirected=True):
    """从 (from, to) 的列表出发，用 BFS 构造最短路径"""
    neighbors = {}
    for p, q in edges:
        neighbors.setdefault(p, []).append(q)
        if undirected:
            neighbors.setdefault(q, []).append(p)

    prev = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == goal:
            break
        for nxt in neighbors.get(cur, []):
            if nxt not in prev:
                prev[nxt] = cur
                queue.append(nxt)

    if goal not in prev:
        return None

    route = [goal]
    while prev[route[-1]] is not None:
        route.append(prev[route[-1]])
    return route[::-1]


print(bfs_route(edges, 1, 6))

[1, 5, 4, 6]


路径是 **1 → 5 → 4 → 6**。`edge(4, 5)` 是无向的，所以也能用于 5→4。

## 例 2：三个水桶问题

有 8 升、5 升、3 升的水桶。最初只有 8 升的桶是满的，其余是空的。要让**两个桶各装 4 升**，该怎么倒水呢？

这可以看成“状态为顶点、倒水操作为边的图的可达性”。先用 Python 把可能的状态和状态之间的转移全部生成出来。

- 水量合计总是 8 升
- 转移只有“从某个桶倒进另一个桶”
- 倒的时候，**目标桶被装满**或者**来源桶被倒空**，二者之一

In [5]:
from itertools import product

sizes = (8, 5, 3)

# 可能的状态（合计 8 升）
states = [(a, b, c)
          for a, b, c in product(*[range(size + 1) for size in sizes])
          if a + b + c == 8]

# 一次倒水能到达的状态对
transitions = []
for s1, s2 in product(states, states):
    changed = [(size, old, new) for size, old, new in zip(sizes, s1, s2) if old != new]
    if len(changed) != 2:
        continue        # 变化的桶正好 2 个（倒出来源和倒进目标）
    if any(new == size or new == 0 for size, old, new in changed):
        transitions.append((s1, s2))    # 目标桶满 or 来源桶空

print(len(states), "种状态，", len(transitions), "种倒法")

24 种状态， 106 种倒法


接下来，把这些转移作为 `pour` 谓词的**事实**注册，再用 `plan` 谓词推理“从初始状态到目标状态有没有步骤”。

- `pour(a, b, c, a2, b2, c2)` … 能从状态 `(a, b, c)` 一步移到 `(a2, b2, c2)`
- `plan(a, b, c, a2, b2, c2)` … 能从状态 `(a, b, c)` 若干步移到 `(a2, b2, c2)`

In [6]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(pour, plan)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

# 一步能移，则 plan 也成立
fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
# 移一步之后还有 plan，则 plan 成立（递归）
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

for s1, s2 in transitions:
    fp.fact(pour(*s1, *s2))

start, goal = (8, 0, 0), (4, 4, 0)

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 0 ns
Wall time: 1.78 ms


结果是 `sat`，所以从 8,0,0 到 4,4,0 有步骤。

步骤可以拿 `transitions`（边的列表）做 BFS 求出。例 1 的 `bfs_route()` 可以直接用（这里是有向的，所以 `undirected=False`）。

In [7]:
route = bfs_route(transitions, start, goal, undirected=False)

for s1, s2 in zip(route, route[1:]):
    print(s1, "->", s2)

(8, 0, 0) -> (3, 5, 0)
(3, 5, 0) -> (3, 2, 3)
(3, 2, 3) -> (6, 2, 0)
(6, 2, 0) -> (6, 0, 2)
(6, 0, 2) -> (1, 5, 2)
(1, 5, 2) -> (1, 4, 3)
(1, 4, 3) -> (4, 4, 0)


用 7 次倒水到达了 4,4,0。人类解需要反复试错的问题，**只要把状态和转移写出来**，剩下的就能机械地解出。

也可以确认注册的规则和事实。

In [8]:
for rule in fp.get_rules()[:5]:
    print(rule)
print("...")

ForAll([A, B, C, D, E, F],
       Implies(pour(A, B, C, D, E, F),
               plan(A, B, C, D, E, F)))
ForAll([A, B, C, D, E, F, G, H, I],
       Implies(And(plan(D, E, F, G, H, I),
                   pour(A, B, C, D, E, F)),
               plan(A, B, C, G, H, I)))
pour(0, 5, 3, 3, 5, 0)
pour(0, 5, 3, 5, 0, 3)
pour(1, 4, 3, 0, 5, 3)
...


## 例 3：把 `pour` 写成规则

例 2 里用 Python 算出了所有转移，把 `pour` 作为事实注册。这次**用规则来写 `pour`**。把可能的状态 `state(a, b, c)` 作为事实给出，把倒水的条件写成规则。

条件是“变化的桶有 2 个”，且其中一个变满或变空。

In [9]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
state = Function('state', s, s, s, BoolSort())
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(state, pour, plan)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

# 倒水规则：变化的桶有 2 个，其中一个变满或变空
fp.rule(
    pour(a, b, c, a1, b1, c1),
    [
        state(a, b, c),
        state(a1, b1, c1),
        Or(
            And(a == a1, b != b1, Or(b1 == 0, b1 == 5, c1 == 0, c1 == 3)),
            And(b == b1, a != a1, Or(a1 == 0, a1 == 8, c1 == 0, c1 == 3)),
            And(c == c1, a != a1, Or(a1 == 0, a1 == 8, b1 == 0, b1 == 5)),
        ),
    ],
)

# 把可能的状态（合计 8 升）注册为事实
for p in states:
    fp.fact(state(*p))

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 62.5 ms
Wall time: 47.1 ms


和例 2 一样是 `sat`。即使用规则写 `pour`，只要把状态作为事实给出，结论相同。本环境里大约 0.04 秒（例 2 大约 1 毫秒）。

读 `pour` 的规则时，要点是本体里放了 `state(a, b, c)` 和 `state(a1, b1, c1)` **两个**。倒完后的状态也必须是可能的状态。

## 例 4：把 `state` 也写成规则

最后把 `state` 也写成规则。把每个桶可取的水量作为 `bucket_a`、`bucket_b`、`bucket_c` 的事实给出，`state` 由它们的组合（合计 8 升）推导出来。

In [10]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
bucket_a = Function('bucket_a', s, BoolSort())
bucket_b = Function('bucket_b', s, BoolSort())
bucket_c = Function('bucket_c', s, BoolSort())
state = Function('state', s, s, s, BoolSort())
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(state, pour, plan, bucket_a, bucket_b, bucket_c)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

fp.rule(
    pour(a, b, c, a1, b1, c1),
    [
        state(a, b, c),
        state(a1, b1, c1),
        Or(
            And(a == a1, b != b1, Or(b1 == 0, b1 == 5, c1 == 0, c1 == 3)),
            And(b == b1, a != a1, Or(a1 == 0, a1 == 8, c1 == 0, c1 == 3)),
            And(c == c1, a != a1, Or(a1 == 0, a1 == 8, b1 == 0, b1 == 5)),
        ),
    ],
)

# state 是“各桶水量的组合”，且合计为 8 升
fp.rule(state(a, b, c), [bucket_a(a), bucket_b(b), bucket_c(c), a + b + c == 8])

for i in range(sizes[0] + 1):
    fp.fact(bucket_a(i), str(i))
for i in range(sizes[1] + 1):
    fp.fact(bucket_b(i), str(i))
for i in range(sizes[2] + 1):
    fp.fact(bucket_c(i), str(i))

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 844 ms
Wall time: 832 ms


这里也是 `sat`，但**花的时间长得多**。本环境里是 0.6 秒，和例 3 的 0.04 秒相比大约 15 倍。

把 `state` 写成规则后，每次应用 `pour` 的规则都要重新考虑 `state` 的组合，搜索量就相应增加。“能用规则写”和“快”是两回事，这个例子说明**已知的关系作为事实给出更快**。

## 小结

| 用的东西 | 作用 |
|---|---|
| `Fixedpoint()` | 创建引擎 |
| `set(engine='datalog')` | 选择 Datalog 引擎 |
| `Function(名称, 类型..., BoolSort())` | 创建谓词 |
| `register_relation(...)` | 把谓词注册到引擎 |
| `declare_var(...)` | 声明规则里用的变量 |
| `rule(头, 本体)` | 添加“本体成立则头成立”的规则 |
| `fact(...)` | 添加事实 |
| `query(...)` | 询问是否成立（`sat` / `unsat`） |
| `get_rules()` | 确认注册的规则和事实 |

- 可达性用“一步能移”+“递归”两条规则就能写
- Fixedpoint 回答的是“能否到达”。具体的路径，把边列表做 BFS 就能构造
- 用说明（`generate_explanations`）取路径的功能在最近的 Z3 里不稳定，最好不要依赖
- 状态和转移已知时，作为事实给出更快。全部写成规则虽然灵活，但会变慢